# Cross-Source Searchlight Decoding (Avg voxel+trial+source z)

**Cross-source comparisons:**
1. Train pleasant vs neutral -> Test pleasantAI vs neutralAI
2. Train pleasantAI vs neutralAI -> Test pleasant vs neutral
3. Train unpleasant vs neutral -> Test unpleasantAI vs neutralAI
4. Train unpleasantAI vs neutralAI -> Test unpleasant vs neutral

**Z-score mode: `voxel_trial_source_zscore`**
- Source-level normalization before searchlight: pattern norm (across voxels per trial) + voxel-wise z (across trials within source)
- Inside each decoding call: pattern norm -> voxel-wise z (using training stats), then pseudo-trial averaging

**Decoding: Avg(random) cross-source**

In [1]:
import os, gc, warnings
import numpy as np
import pandas as pd
import scipy.io
import nibabel as nib
from nilearn.image import resample_img
from rsatoolbox.util.searchlight import get_volume_searchlight
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from scipy.stats import zscore
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import pickle

warnings.filterwarnings('ignore')

## Configuration

In [2]:
beta_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\GLM_singletrial\betas'
label_file = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\GLM_singletrial\beta_groups.csv'
mask_file = r'N:\Experimental_Data\yujunchen\projects\data\masks\MNI152_T1_2mm_brain_mask.nii.gz'
onset_base_dir = r'N:\Experimental_Data\yujunchen\projects\LAB_IAPS_AI\DataRecording'

subs = ['Sub1', 'Sub2', 'Sub3', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub8', 'Sub9',
        'Sub11', 'Sub12', 'Sub13', 'Sub14', 'Sub15', 'Sub16', 'Sub17', 'Sub18',
        'Sub19', 'Sub20', 'Sub21', 'Sub22', 'Sub23', 'Sub24', 'Sub25', 'Sub26',
        'Sub27', 'Sub28', 'Sub29', 'Sub30', 'Sub31']

runs = ['Run01','Run02','Run03','Run04','Run05','Run06','Run07','Run08','Run09','Run10']

# Searchlight parameters
RADIUS = 5          # mm
N_REPEATS = 20      # CV repeats for Avg(random)
N_FOLDS = 4         # pseudo-trial groups
N_AVG_GROUPS = 3    # training pseudo-trial groups
N_JOBS = 6          # parallel subjects
MIN_VOXELS = 10     # minimum valid voxels per searchlight

natural_cats = ['pleasant', 'neutral', 'unpleasant']
ai_cats = ['pleasantAI', 'neutralAI', 'unpleasantAI']
all_categories = natural_cats + ai_cats

cross_comparisons = [
    ('pleasant', 'neutral', 'pleasantAI', 'neutralAI'),
    ('pleasantAI', 'neutralAI', 'pleasant', 'neutral'),
    ('unpleasant', 'neutral', 'unpleasantAI', 'neutralAI'),
    ('unpleasantAI', 'neutralAI', 'unpleasant', 'neutral'),
]

output_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_cross_source'
os.makedirs(output_dir, exist_ok=True)

## Build Stimulus -> Category Lookup

In [3]:
_bl = pd.read_csv(label_file, header=None, names=['beta_file', 'category'])
beta_file_list = _bl['beta_file'].tolist()

_stim = []
for run_name in runs:
    _mat = scipy.io.loadmat(os.path.join(onset_base_dir, 'Sub27', 'LogFiles', f'{run_name}.mat'), squeeze_me=False)
    _raw = _mat['dataLog']
    for _row in range(1, _raw.shape[0]):
        _c = _raw[_row, 1]
        if _c.size == 0: continue
        if str(_c.flat[0]).strip() == 'Stim on':
            _stim.append(str(_raw[_row, 2].flat[0]).strip())

_cats = _bl['category'].tolist()
stim_to_category = {s: c for s, c in zip(_stim, _cats)}
print(f"Mapped {len(stim_to_category)} stimuli to categories")

Mapped 120 stimuli to categories


## Helper Functions

In [4]:
def safe_pattern_zscore(X):
    """Pattern normalization: z-score across voxels within each trial."""
    Xz = zscore(X, axis=1)
    return np.nan_to_num(Xz, nan=0.0, posinf=0.0, neginf=0.0)


def fit_trial_zscore(X_train):
    """Fit voxel-wise standardization parameters across trials."""
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1
    return mean, std


def apply_trial_zscore(X, mean, std):
    """Apply voxel-wise standardization."""
    return (X - mean) / std


def compute_voxel_source_zscore(cond_data_dict):
    """Source-level z-scoring: pattern norm per trial, then voxel-wise z within source.
    Applied before searchlight to keep Natural and AI sources separate."""
    result = {}
    for source_cats in [natural_cats, ai_cats]:
        present = [c for c in source_cats if c in cond_data_dict]
        if not present:
            continue
        all_data = np.vstack([cond_data_dict[c] for c in present])
        sizes = [len(cond_data_dict[c]) for c in present]
        # Step 1: pattern norm
        all_z = safe_pattern_zscore(all_data)
        # Step 2: voxel-wise z within source
        m = np.mean(all_z, axis=0)
        s = np.std(all_z, axis=0)
        s[s == 0] = 1
        all_z = (all_z - m) / s
        idx = 0
        for c, sz in zip(present, sizes):
            result[c] = all_z[idx:idx+sz]
            idx += sz
    return result


def average_chunks(X, chunks):
    """Average trials within chunk groups to create pseudo-trials."""
    return np.array([np.mean(X[ch], axis=0) for ch in chunks])


def preprocess_voxel_trial_inside_fold(X_train, X_test):
    """Inside-fold preprocessing for voxel_trial_source_zscore:
    pattern norm -> voxel-wise z using training fold stats."""
    X_train = safe_pattern_zscore(X_train)
    X_test = safe_pattern_zscore(X_test)
    mean, std = fit_trial_zscore(X_train)
    X_train = apply_trial_zscore(X_train, mean, std)
    X_test = apply_trial_zscore(X_test, mean, std)
    return X_train, X_test

## Searchlight Cross-Source Decoding Function

In [5]:
def decode_cross_searchlight_vts(train_d1, train_d2, test_d1, test_d2,
                                  n_repeats=N_REPEATS, n_folds=N_FOLDS):
    """Cross-source Avg(random) decoding for a single searchlight sphere.
    
    Data is already voxel_source_z normalized.
    Inside each repeat: pattern norm -> trial z (train stats) -> pseudo-trial averaging.
    """
    accuracies = []
    for repeat in range(n_repeats):
        # Preprocess: pattern norm -> trial z inside this repeat
        train_all = np.vstack([train_d1, train_d2])
        test_all = np.vstack([test_d1, test_d2])
        train_all, test_all = preprocess_voxel_trial_inside_fold(train_all, test_all)

        n_tr1 = len(train_d1)
        n_te1 = len(test_d1)

        # Random grouping for pseudo-trial averaging
        perm = np.random.RandomState(42 + repeat).permutation(n_tr1)
        groups = np.array_split(perm, n_folds)

        td1a = average_chunks(train_all[:n_tr1], groups)
        td2a = average_chunks(train_all[n_tr1:], groups)
        te1a = average_chunks(test_all[:n_te1], groups)
        te2a = average_chunks(test_all[n_te1:], groups)

        X_train = np.vstack([td1a, td2a])
        y_train = np.array([1]*len(td1a) + [0]*len(td2a))
        X_test = np.vstack([te1a, te2a])
        y_test = np.array([1]*len(te1a) + [0]*len(te2a))

        clf = SVC(kernel='linear', C=1.0)
        clf.fit(X_train, y_train)
        accuracies.append(clf.score(X_test, y_test))

    return np.mean(accuracies)

## Setup Mask & Searchlight Grid

In [6]:
# Load mask
mask_img = nib.load(mask_file)

# Use a beta file as template for resampling
temp_nii = nib.load(os.path.join(beta_dir, 'Sub1', 'beta_0001.nii'))
resampled_mask = resample_img(mask_img, target_affine=temp_nii.affine, target_shape=temp_nii.shape)
mask_data = resampled_mask.get_fdata()
dims = mask_data.shape
n_voxels_total = int(np.prod(dims))

print(f"Brain dimensions: {dims}")
print(f"Total voxels: {n_voxels_total}")
print(f"In-mask voxels: {int(mask_data.sum())}")

# Build searchlight grid
print(f"\nCreating searchlight grid (radius={RADIUS}mm)...")
centers, neighbors = get_volume_searchlight(mask_data, radius=RADIUS, threshold=0.5)
print(f"Number of searchlight centers: {len(centers)}")

Brain dimensions: (79, 95, 79)
Total voxels: 592895
In-mask voxels: 228419

Creating searchlight grid (radius=5mm)...


Finding searchlights...: 100%|██████████| 228419/228419 [00:39<00:00, 5762.63it/s]


Found 223745 searchlights
Number of searchlight centers: 223745


## Main Analysis: Per-Subject Searchlight

In [7]:
def process_one_subject_searchlight(sub):
    """Process all cross-source searchlight decoding for one subject."""

    # --- Load onsets ---
    sub_onset_dir = os.path.join(onset_base_dir, sub, 'LogFiles')
    stim_names = []
    for run_name in runs:
        fpath = os.path.join(sub_onset_dir, f'{run_name}.mat')
        if not os.path.exists(fpath):
            alt_name = 'Run' + str(int(run_name.replace('Run', '')))
            fpath = os.path.join(sub_onset_dir, f'{alt_name}.mat')
        mat = scipy.io.loadmat(fpath, squeeze_me=False)
        raw = mat['dataLog']
        for row in range(1, raw.shape[0]):
            c = raw[row, 1]
            if c.size == 0:
                continue
            if str(c.flat[0]).strip() == 'Stim on':
                stim_names.append(str(raw[row, 2].flat[0]).strip())

    categories = [stim_to_category[s] for s in stim_names]
    bl = pd.DataFrame({'beta_file': beta_file_list, 'category': categories, 'stimulus': stim_names})

    # --- Load all 600 betas ---
    sub_dir = os.path.join(beta_dir, sub)
    first_img = nib.load(os.path.join(sub_dir, bl['beta_file'].iloc[0]))
    n_vox = int(np.prod(first_img.shape))
    data = np.zeros((600, n_vox), dtype=np.float32)
    for idx in range(600):
        data[idx] = nib.load(os.path.join(sub_dir, bl.iloc[idx]['beta_file'])).get_fdata().flatten().astype(np.float32)

    # --- Extract condition data ---
    cond_data = {}
    for cond in all_categories:
        idx_arr = bl[bl['category'] == cond].index.values
        cond_data[cond] = data[idx_arr]

    # --- Source-level z-scoring (all voxels, before searchlight) ---
    voxel_source_z = compute_voxel_source_zscore(cond_data)

    del data, cond_data
    gc.collect()

    # --- Results: one accuracy map per comparison ---
    comp_maps = {}
    for train_c1, train_c2, test_c1, test_c2 in cross_comparisons:
        comp = f"train_{train_c1}v{train_c2}_test_{test_c1}v{test_c2}"
        comp_maps[comp] = np.full(n_voxels_total, np.nan, dtype=np.float32)

    # --- Searchlight loop ---
    for center_idx in range(len(centers)):
        center_flat_idx = centers[center_idx]
        nb_idx = neighbors[center_idx]

        if len(nb_idx) < MIN_VOXELS:
            continue

        # Check NaN on one condition as proxy
        sample_data = voxel_source_z[all_categories[0]][:, nb_idx]
        valid_vox = ~np.any(np.isnan(sample_data), axis=0)
        if valid_vox.sum() < MIN_VOXELS:
            continue

        if valid_vox.all():
            sphere_vox = nb_idx
        else:
            sphere_vox = nb_idx[np.where(valid_vox)[0]]

        for train_c1, train_c2, test_c1, test_c2 in cross_comparisons:
            comp = f"train_{train_c1}v{train_c2}_test_{test_c1}v{test_c2}"

            td1 = voxel_source_z[train_c1][:, sphere_vox]
            td2 = voxel_source_z[train_c2][:, sphere_vox]
            ted1 = voxel_source_z[test_c1][:, sphere_vox]
            ted2 = voxel_source_z[test_c2][:, sphere_vox]

            # Check variance
            if np.std(td1) == 0 or np.std(ted1) == 0:
                continue

            try:
                acc = decode_cross_searchlight_vts(td1, td2, ted1, ted2)
                comp_maps[comp][center_flat_idx] = acc
            except Exception:
                continue

    # Print progress
    for comp, acc_map in comp_maps.items():
        valid = ~np.isnan(acc_map)
        if valid.sum() > 0:
            print(f"  {sub} | {comp}: mean={acc_map[valid].mean():.3f}, "
                  f"max={acc_map[valid].max():.3f}, "
                  f">50%: {(acc_map[valid]>0.5).sum()}/{valid.sum()}")

    return {'sub': sub, 'maps': comp_maps}

## Run Searchlight Analysis

In [8]:
print(f"Running cross-source searchlight for {len(subs)} subjects...")
print(f"Searchlight: radius={RADIUS}mm, centers={len(centers)}")
print(f"Decoding: Avg(random), {N_REPEATS} repeats, {N_FOLDS} groups")
print(f"Z-mode: voxel_trial_source_zscore")
print(f"Parallel jobs: {N_JOBS}")
print()

results_list = Parallel(n_jobs=N_JOBS, prefer='threads', verbose=10)(
    delayed(process_one_subject_searchlight)(sub) for sub in subs
)

# Organize results
searchlight_results = {}
for train_c1, train_c2, test_c1, test_c2 in cross_comparisons:
    comp = f"train_{train_c1}v{train_c2}_test_{test_c1}v{test_c2}"
    searchlight_results[comp] = {}

for r in results_list:
    sub = r['sub']
    for comp, acc_map_flat in r['maps'].items():
        searchlight_results[comp][sub] = acc_map_flat.reshape(dims)

print(f"\nAll {len(subs)} subjects complete.")

Running cross-source searchlight for 30 subjects...
Searchlight: radius=5mm, centers=223745
Decoding: Avg(random), 20 repeats, 4 groups
Z-mode: voxel_trial_source_zscore
Parallel jobs: 6



[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed: 19.8min
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed: 20.0min
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed: 61.0min
[Parallel(n_jobs=6)]: Done  23 out of  30 | elapsed: 81.2min remaining: 24.7min
[Parallel(n_jobs=6)]: Done  27 out of  30 | elapsed: 100.3min remaining: 11.1min



All 30 subjects complete.


[Parallel(n_jobs=6)]: Done  30 out of  30 | elapsed: 100.5min finished


In [14]:
searchlight_results['train_pleasantAIvneutralAI_test_pleasantvneutral']['Sub1'].shape

(79, 95, 79)

## Average Across Subjects & Save

In [9]:
print("Averaging across subjects and saving NIfTI files...")
print(f"Output: {output_dir}\n")

summary = {}

for comp in searchlight_results:
    sub_maps = []
    for sub in subs:
        if sub in searchlight_results[comp]:
            sub_maps.append(searchlight_results[comp][sub])

    all_maps = np.stack(sub_maps, axis=0)
    avg_map = np.nanmean(all_maps, axis=0)
    std_map = np.nanstd(all_maps, axis=0)
    n_valid = np.sum(~np.isnan(all_maps), axis=0)

    summary[comp] = {'mean': avg_map, 'std': std_map, 'n': n_valid}

    # Save mean accuracy map
    avg_nii = nib.Nifti1Image(avg_map, resampled_mask.affine, resampled_mask.header)
    nib.save(avg_nii, os.path.join(output_dir, f"{comp}_mean.nii.gz"))

    # Save std map
    std_nii = nib.Nifti1Image(std_map, resampled_mask.affine, resampled_mask.header)
    nib.save(std_nii, os.path.join(output_dir, f"{comp}_std.nii.gz"))

    # Save subject count map
    n_nii = nib.Nifti1Image(n_valid.astype(np.float32), resampled_mask.affine, resampled_mask.header)
    nib.save(n_nii, os.path.join(output_dir, f"{comp}_nsubjects.nii.gz"))

    # Save thresholded maps
    for thresh in [0.53, 0.55, 0.60]:
        thresh_map = np.copy(avg_map)
        thresh_map[avg_map <= thresh] = np.nan
        if np.any(~np.isnan(thresh_map)):
            t_nii = nib.Nifti1Image(thresh_map, resampled_mask.affine, resampled_mask.header)
            nib.save(t_nii, os.path.join(output_dir, f"{comp}_thresh{int(thresh*100)}.nii.gz"))

    # Print summary
    valid_accs = avg_map[~np.isnan(avg_map)]
    if len(valid_accs) > 0:
        print(f"{comp}:")
        print(f"  Subjects: {len(sub_maps)}")
        print(f"  Valid searchlights: {len(valid_accs)}")
        print(f"  Mean accuracy: {valid_accs.mean():.3f} +/- {valid_accs.std():.3f}")
        print(f"  Max accuracy: {valid_accs.max():.3f}")
        print(f"  Voxels > 0.50: {(valid_accs > 0.50).sum()} ({(valid_accs > 0.50).mean()*100:.1f}%)")
        print(f"  Voxels > 0.55: {(valid_accs > 0.55).sum()} ({(valid_accs > 0.55).mean()*100:.1f}%)")
        print()

# Save everything as pickle
all_results = {
    'searchlight_results': searchlight_results,
    'summary': summary,
    'subs': subs,
    'dims': dims,
    'cross_comparisons': cross_comparisons,
    'z_mode': 'voxel_trial_source_zscore',
    'radius': RADIUS,
    'n_repeats': N_REPEATS,
    'n_folds': N_FOLDS,
}
pkl_path = os.path.join(output_dir, 'searchlight_cross_source_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(all_results, f)
print(f"Saved all results to {pkl_path}")

Averaging across subjects and saving NIfTI files...
Output: N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_cross_source

Saved all results to N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_cross_source\searchlight_cross_source_results.pkl


## Save Individual Subject Maps

In [10]:
sub_output_dir = os.path.join(output_dir, 'per_subject')
os.makedirs(sub_output_dir, exist_ok=True)

for comp in searchlight_results:
    for sub in searchlight_results[comp]:
        acc_map_3d = searchlight_results[comp][sub]
        nii = nib.Nifti1Image(acc_map_3d, resampled_mask.affine, resampled_mask.header)
        nib.save(nii, os.path.join(sub_output_dir, f"{comp}_{sub}.nii.gz"))

print(f"Saved individual subject maps to {sub_output_dir}")

Saved individual subject maps to N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_cross_source\per_subject


## Peak Accuracy Locations

In [11]:
for comp in summary:
    avg_map = summary[comp]['mean']
    valid = ~np.isnan(avg_map)
    if valid.sum() == 0:
        continue

    flat = avg_map.flatten()
    top_idx = np.nanargmax(flat)
    peak_coords_voxel = np.unravel_index(top_idx, dims)
    peak_acc = flat[top_idx]

    # Convert to MNI coordinates
    mni_coords = nib.affines.apply_affine(resampled_mask.affine, peak_coords_voxel)

    print(f"{comp}:")
    print(f"  Peak accuracy: {peak_acc:.3f}")
    print(f"  Voxel coords: {peak_coords_voxel}")
    print(f"  MNI coords: ({mni_coords[0]:.0f}, {mni_coords[1]:.0f}, {mni_coords[2]:.0f})")
    print()